In [4]:
# ==============================
# RNA Sequence Predictor (Standalone)
# ==============================
from Bio import SeqIO
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
import os

# ==============================
# 1. Input FASTA sequences
# ==============================
input_fasta = "/input_sequences.txt"  # <-- your RNA sequences
sequences = [str(record.seq) for record in SeqIO.parse(input_fasta, "fasta")]
sequence_ids = [record.id for record in SeqIO.parse(input_fasta, "fasta")]

print(f"Loaded {len(sequences)} sequences for prediction")

# ==============================
# 2. Load tokenizer and trained model
# ==============================
model_path = r"C:\huggingface_models\nucleotide_transformer"
checkpoint_path = r"./results/non_overlapping/checkpoint-9894/pytorch_model.bin"

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    trust_remote_code=True,
    state_dict=torch.load(checkpoint_path, map_location="cpu")
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# ==============================
# 3. Dataset class for prediction
# ==============================
class RNA6MerDataset(Dataset):
    def __init__(self, sequences, tokenizer, k=6, max_length=1000):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.k = k
        self.max_length = max_length

    def split_kmers(self, seq):
        seq = seq.upper().replace("U", "T")
        if len(seq) < self.k:
            seq += "N" * (self.k - len(seq))
        kmers = [seq[i:i+self.k] for i in range(0, len(seq), self.k)]
        if len(kmers[-1]) < self.k:
            kmers[-1] += "N" * (self.k - len(kmers[-1]))
        return kmers

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        kmers = self.split_kmers(self.sequences[idx])
        tokens = self.tokenizer(
            kmers[:998],  # 998 + <cls> + <eos> ≈ 1000 tokens
            is_split_into_words=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in tokens.items()}
        return item

# ==============================
# 4. Create dataset and dataloader
# ==============================
predict_dataset = RNA6MerDataset(sequences, tokenizer)
predict_loader = DataLoader(predict_dataset, batch_size=8, shuffle=False)

# ==============================
# 5. Prediction loop
# ==============================
all_preds = []
all_probs = []

with torch.no_grad():
    for batch in predict_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=-1)[:, 1]  # probability of positive class
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# ==============================
# 6. Print predictions
# ==============================

import pandas as pd

# Map numeric predictions to descriptive labels
pred_labels = {0: "Non-phase-separated RNA", 1: "Phase-separated RNA"}
mapped_preds = [pred_labels[p] for p in all_preds]

# Create DataFrame
pred_df = pd.DataFrame({
    "Sequence_ID": sequence_ids,
    "Prediction": mapped_preds,
    "Probability": [round(p, 4) for p in all_probs]
})

# Display the table
print(pred_df)

#  save to CSV
pred_df.to_csv("rna_predictions.csv", index=False)

Loaded 20 sequences for prediction
   Sequence_ID               Prediction  Probability
0            1      Phase-separated RNA       0.9994
1            2      Phase-separated RNA       0.9975
2            3      Phase-separated RNA       0.9985
3            4      Phase-separated RNA       0.9989
4            5      Phase-separated RNA       0.9848
5            6      Phase-separated RNA       0.9948
6            7      Phase-separated RNA       0.9870
7            8      Phase-separated RNA       0.9925
8            9      Phase-separated RNA       0.9980
9           10      Phase-separated RNA       0.9995
10          11  Non-phase-separated RNA       0.0008
11          12  Non-phase-separated RNA       0.0134
12          13  Non-phase-separated RNA       0.0009
13          14  Non-phase-separated RNA       0.0006
14          15  Non-phase-separated RNA       0.0005
15          16  Non-phase-separated RNA       0.0008
16          17  Non-phase-separated RNA       0.0007
17         

In [3]:
!conda env export > environment.yml